In [ ]:
import os

import ee

from utils import (
    get_export_periods,
    get_export_tile_ids,
    get_wgs84_region_bounds,
    get_region_pixel_size,
    get_region_crs,
    get_region_transform,
    gee_grid_tiles,
    get_long_region_name,
    get_long_ds_name,
    get_nodata_value
)
from speckle_filter import RefinedLee

First, we must retrieve cloud bucket access credentials from the corresponding JSON file.

In [ ]:
# Replace with your equivalent path to credentials
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "path/to/your/credentials.json"
)

In [ ]:
ee.Authenticate()

In [ ]:
# Replace with your equivalent cloud project
ee.Initialize(project="your-cloud-project")

# Input arguments

In [ ]:
# Bucket name
bucket = "bucket_name"

# Region
short_region_name = "est"

# Dataset
short_ds_name = "s1"

Uncomment all years in order to process the whole time series.

In [ ]:
# List of years to process
years = [
    # 2015,
    # 2016,
    # 2017,
    # 2018,
    # 2019,
    # 2020,
    # 2021,
    # 2022,
    # 2023,
    # 2024,
    2025
]

In [ ]:
# Dictionary of time periods to export
export_periods = get_export_periods(years, short_region_name)

In [ ]:
print(export_periods)

You can uncomment the next cell if you wish to test the workflow with a single season instead of the full list.

In [ ]:
export_periods = {}
for year in years:
    export_periods[year] = {
        "start_dates": [
            # f"{year}-04-01",
            f"{year}-06-01",
            # f"{year}-09-01"
        ],
        "end_dates": [
            # f"{year}-05-31",
            f"{year}-08-31",
            # f"{year}-10-31"
        ]
    }

Uncomment the variables you would like to process.

In [ ]:
# List of variables to calculate
variables = [
    "rvi",
    # "vvvhr"
]

In [ ]:
# List of statistics to calculate
stats = [
    "median",
    # "mean",
    # "std",
    # "min",
    # "max"
]

In [ ]:
# Dictionary mapping statistic keys to reducer functions
stat_reducers = {
    "median": lambda col: col.median(),
    "mean": lambda col: col.mean(),
    "std": lambda col: col.reduce(ee.Reducer.stdDev()),
    "min": lambda col: col.min(),
    "max": lambda col: col.max()
}

Tile IDs that are going to be exported are given in the following list. Only the tiles that intersect with the boundaries of the corresponding region (Estonia, Baltic states or European countries) will be exported.

In [ ]:
# Tile IDs to export
export_tile_ids = get_export_tile_ids(short_region_name)

In [ ]:
print(export_tile_ids)

You can uncomment the next cell if you wish to test the workflow with a single tile or a couple of tiles instead of the full list.

In [ ]:
export_tile_ids = ["09"]

# Create output grid tiles

We will create an output grid where each tile is 10000$\times$10000 pixels, which is the default maximum output image size in GEE. This means that for 10 m resolution the `scale` parameter will be 100000 (100 km), for 30 m resolution 300000 (300 km) and for 100 m resolution 1000000 (1000 km).

In [ ]:
# Region bounds in WGS84
region_bounds = get_wgs84_region_bounds(short_region_name)

# Pixel size
pixel_size = get_region_pixel_size(short_region_name)

# Region CRS
region_crs = get_region_crs(short_region_name)

# Calculate transform
region_transform = get_region_transform(short_region_name)

# Generate grid tiles
grid_tiles = gee_grid_tiles(region_bounds, region_crs, pixel_size)

In [ ]:
print(region_crs)
print(region_transform)

# Calculate indices and export tiles

In [ ]:
# Convert from dB to linear scale
def db_to_power(image):
    return ee.Image(10).pow(image.divide(10))

In [ ]:
# Convert from linear scale to dB
def power_to_db(image):
    return image.log10().multiply(10)

In [ ]:
# Apply preprocessing steps to Sentinel-1 collection
def preprocess_s1_collection(start_date, end_date, grid_tiles, variable):
    
    # Define the required bands for each index
    required_bands = {
        "rvi": ["VV", "VH"],
        "vvvhr": ["VV", "VH"]
    }
    
    # Read Sentinel-1 collection
    s1 = ee.ImageCollection("COPERNICUS/S1_GRD")
    
    # Filter Sentinel-1 based on time and bounds
    s1_filtered = s1\
        .filterDate(
            ee.Date(start_date), ee.Date(end_date).advance(1, "day")
        )\
        .filterBounds(grid_tiles.geometry())\
        .filter(
            ee.Filter.listContains("transmitterReceiverPolarisation", "VV")
        )\
        .filter(
            ee.Filter.listContains("transmitterReceiverPolarisation", "VH")
        )\
        .filter(ee.Filter.eq("instrumentMode", "IW"))\
        .filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))

    # Set the desired bands based on the requested index
    desired_bands = required_bands.get(variable.lower(), [])
    s1_filtered_reordered = s1_filtered\
        .map(lambda image: image.select(desired_bands))
    
    # Convert to linear scale
    s1_linear = s1_filtered_reordered.map(db_to_power)
    
    # Apply speckle filter
    s1_no_speckle = s1_linear.map(RefinedLee)
    
    # Convert back to dB
    s1_no_speckle_db = s1_no_speckle.map(power_to_db)
    
    return s1_no_speckle_db

In [ ]:
# Calculate index based on variable name for Sentinel-1 collection
def calc_s1_index_band(s1_collection, variable):
    
    # Normalize input to lowercase for consistency
    variable = variable.lower()
    
    # Radar Vegetation Index (RVI)
    if variable == "rvi":
        
        # Convert VV and VH from dB to linear
        vv_linear = db_to_power(s1_collection.select("VV"))
        vh_linear = db_to_power(s1_collection.select("VH"))
        
        # Compute RVI in linear scale
        index_band = (
            vh_linear
            .multiply(4)
            .divide(vv_linear.add(vh_linear))
        ).rename(variable)
        
    # VV/VH index
    elif variable == "vvvhr":
        
        # Convert VV and VH from dB to linear
        vv_linear = db_to_power(s1_collection.select("VV"))
        vh_linear = db_to_power(s1_collection.select("VH"))
        
        # Calculate VV/VH in linear scale and convert to dB
        index_band = power_to_db(
            vv_linear
            .divide(vh_linear)
        ).rename(variable)
        
    return index_band

In [ ]:
# List of grid IDs
grid_ids = grid_tiles.aggregate_array("system:index").getInfo()

In [ ]:
# Load land mask from asset
if short_region_name == "est":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "est_ref_grid_500m_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)
elif short_region_name == "bal":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "bal_ref_grid_1km_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)
elif short_region_name == "eur":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "eur_ref_grid_1km_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)

In [ ]:
# Loop over years
for year in export_periods.keys():
    
    print(f"Year being processed: {year}")

    # Start dates
    start_dates = export_periods[year]["start_dates"]

    # End dates
    end_dates = export_periods[year]["end_dates"]

    # Loop over variables
    for variable in variables:
        
        print(f"\tVariable being processed: {variable}")

        # Nodata value
        nodata_val = get_nodata_value(variable, short_region_name)

        # Preprocess annual collection
        s1_preprocessed_annual = preprocess_s1_collection(
            f"{year}-01-01", f"{year}-12-31", grid_tiles, variable
        )

        # Resample annual collection based on region
        if short_region_name in ["bal", "eur"]:
            if variable != "wiw":
                s1_preprocessed_annual = (
                    s1_preprocessed_annual
                    .map(lambda img: img.resample("bilinear"))
                )

        # Calculate index per image in annual collection
        index_collection_annual = (
            s1_preprocessed_annual
            .map(lambda img: calc_s1_index_band(img, variable))
            .select(variable)
        )

        # Loop over statistics
        for stat in stats:

            print(f"\t\tStatistic being processed: {stat}")
            
            # Get reducer function
            reducer_fn = stat_reducers[stat]

            # Calculate annual statistic
            index_annual_stat = (
                reducer_fn(index_collection_annual)
                .rename(f"{variable}_{stat}")
                .unmask(value=nodata_val)
            )

            # Loop over time periods
            for start_date, end_date in zip(start_dates, end_dates):
                
                print(f"\t\t\tTime period start date: {start_date}")
                print(f"\t\t\tTime period end date: {end_date}")

                # Preprocess seasonal collection
                s1_preprocessed = preprocess_s1_collection(
                    start_date, end_date, grid_tiles, variable
                )

                # Resample seasonal collection based on region
                if short_region_name in ["bal", "eur"]:
                    if variable != "wiw":
                        s1_preprocessed = (
                            s1_preprocessed
                            .map(lambda img: img.resample("bilinear"))
                        )

                # Calculate index per image in seasonal collection
                index_collection_seasonal = (
                    s1_preprocessed
                    .map(lambda img: calc_s1_index_band(img, variable))
                    .select(variable)
                )

                # Calculate seasonal statistic
                index_season_stat = (
                    reducer_fn(index_collection_seasonal)
                    .rename(f"{variable}_{stat}")
                )

                # Create a mask for nodata pixels
                nodata_mask = (
                    index_season_stat
                    .unmask(value=nodata_val)
                    .eq(nodata_val)
                )
                
                # Fill masked pixels with annual values
                image_filled = (
                    index_season_stat
                    .unmask(index_annual_stat)
                    .where(
                        index_season_stat.eq(nodata_val),
                        index_annual_stat
                    )
                )

                # Replace masked pixels with the nodata value
                unmasked_image = image_filled.unmask(value=nodata_val)
                
                # Scale values
                out_image = (
                    unmasked_image.where(nodata_mask, 0)
                    .multiply(1000)
                    .round()
                )
                if variable == "vvvhr":
                    out_image = out_image.toInt32()
                else:
                    out_image = out_image.toInt16()
                
                # Apply land mask
                if short_region_name in ["est", "bal", "eur"]:
                    out_image = (
                        out_image.where(nodata_mask, nodata_val)
                        .updateMask(land_mask)
                        .unmask(nodata_val)
                    )
                else:
                    out_image = out_image.where(nodata_mask, nodata_val)
                    
                # Loop over grid tile IDs
                for i, grid_id in enumerate(grid_ids):

                    # Output tile ID
                    tile_id = str(i + 1).zfill(2)
                    
                    if tile_id in export_tile_ids:

                        # Get tile geometry
                        feature = ee.Feature(grid_tiles.toList(1, i).get(0))
                        region = feature.geometry()
                        
                        # Launch export task for filled image tile
                        filename = "_".join(
                            [
                                short_region_name,
                                short_ds_name,
                                f"{variable}_{stat}",
                                start_date,
                                end_date,
                                tile_id
                            ]
                        )
                        task_name = f"export_{filename}"
                        long_region_name = get_long_region_name(
                            short_region_name
                        )
                        long_ds_name = get_long_ds_name(
                            short_ds_name
                        )
                        prefix = (
                            f"{long_region_name}/"
                            f"{long_ds_name}/"
                            f"{variable}/"
                            f"{year}/"
                            f"{filename}"
                        )
                        task = ee.batch.Export.image.toCloudStorage(
                            image=out_image,
                            description=task_name,
                            bucket=bucket,
                            fileNamePrefix=prefix,
                            fileFormat="GeoTIFF",
                            crs=region_crs,
                            crsTransform=region_transform,
                            maxPixels=1e13,
                            region=region,
                            formatOptions={
                                "cloudOptimized": True,
                                "noData": nodata_val
                            }
                        )
                        task.start()
                        print(f"\t\t\t\tStarted task: {task_name}")
                        
                        # Launch export task for nodata tile
                        # nodata_filename = "_".join(
                        #     [
                        #         short_region_name,
                        #         short_ds_name,
                        #         f"{variable}",
                        #         start_date,
                        #         end_date,
                        #         tile_id,
                        #         "nodata"
                        #     ]
                        # )
                        # nodata_task_name = f"export_{nodata_filename}"
                        # nodata_prefix = (
                        #     f"{long_region_name}_nodata/"
                        #     f"{long_ds_name}/"
                        #     f"{variable}/"
                        #     f"{year}/"
                        #     f"{nodata_filename}"
                        # )
                        # nodata_task = ee.batch.Export.image.toCloudStorage(
                        #     image=nodata_mask,
                        #     description=nodata_task_name,
                        #     bucket=bucket,
                        #     fileNamePrefix=nodata_prefix,
                        #     fileFormat="GeoTIFF",
                        #     crs=region_crs,
                        #     crsTransform=region_transform,
                        #     maxPixels=1e13,
                        #     region=region,
                        #     formatOptions={
                        #         "cloudOptimized": True,
                        #         "noData": 0
                        #     }
                        # )
                        # nodata_task.start()
                        # print(f"\t\t\t\tStarted task: {nodata_task_name}")
                    
    print("\n")